# Super-GM Chess Engine Playground

Topic : GAMING AI
Lecture : Mr Akhona Njeje

A playable notebook powered by **Stockfish NNUE**, `python-chess`, and `ipywidgets`.

**Transparency:** The super-GM strength comes from Stockfish. Databricks/Jupyter provides the interactive Python interface and compute environment.


## 1. Install dependencies

Run once. Restart Python if Databricks requests it.


In [ ]:
%pip install -q python-chess ipywidgets


## 2. Install or locate Stockfish

The cell checks common Linux paths, tries the package manager, then compiles the official source as a fallback.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

CANDIDATES = [shutil.which("stockfish"), "/usr/games/stockfish", "/usr/local/bin/stockfish", "/databricks/driver/Stockfish/src/stockfish"]

def existing_binary():
    for p in CANDIDATES:
        if p and Path(p).exists() and os.access(p, os.X_OK):
            return str(Path(p))
    return None

STOCKFISH_PATH = existing_binary()
if STOCKFISH_PATH is None:
    subprocess.run("apt-get update -qq && apt-get install -y stockfish", shell=True, text=True, capture_output=True)
    STOCKFISH_PATH = existing_binary()
if STOCKFISH_PATH is None:
    subprocess.run(
        "cd /databricks/driver && rm -rf Stockfish && "
        "git clone --depth 1 https://github.com/official-stockfish/Stockfish.git && "
        "cd Stockfish/src && make -j2 build ARCH=x86-64",
        shell=True, check=True
    )
    STOCKFISH_PATH = "/databricks/driver/Stockfish/src/stockfish"
print("Stockfish binary:", STOCKFISH_PATH)
assert Path(STOCKFISH_PATH).exists()


## 3. Start maximum-strength engine


In [ ]:
import chess
import chess.engine
import chess.svg
import ipywidgets as widgets
from IPython.display import display, SVG, clear_output
import os

engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
threads = max(1, min(8, (os.cpu_count() or 2) - 1))
engine.configure({"Threads": threads, "Hash": 512, "Skill Level": 20})
print(f"Engine ready with {threads} thread(s).")
print(engine.id)


## 4. Play

Enter moves in UCI format, such as `e2e4`, `g1f3`, or `e7e8q`. Choose a side, set thinking time, and press **New Game**.


In [ ]:
board = chess.Board()
human_color = chess.WHITE
side = widgets.Dropdown(options=[("Play White", "white"), ("Play Black", "black")], value="white", description="Your side:")
move_time = widgets.FloatSlider(value=1.5, min=0.1, max=10.0, step=0.1, description="Think (sec):", continuous_update=False)
move_input = widgets.Text(value="", placeholder="e2e4", description="Your move:")
new_button = widgets.Button(description="New Game", button_style="success")
play_button = widgets.Button(description="Play Move", button_style="primary")
undo_button = widgets.Button(description="Undo Turn", button_style="warning")
resign_button = widgets.Button(description="Resign", button_style="danger")
status = widgets.HTML()
board_output = widgets.Output()
analysis_output = widgets.Output()

def render_board(message=""):
    with board_output:
        clear_output(wait=True)
        display(SVG(chess.svg.board(board=board, orientation=human_color, size=520)))
    outcome = board.outcome(claim_draw=True)
    if outcome:
        status.value = f"<b>Game over:</b> {outcome.result()} | {outcome.termination.name}"
    else:
        turn = "White" if board.turn else "Black"
        status.value = f"<b>{message}</b><br>Turn: {turn}"

def show_analysis():
    if board.is_game_over(claim_draw=True): return
    with analysis_output:
        clear_output(wait=True)
        infos = engine.analyse(board, chess.engine.Limit(time=min(float(move_time.value), 2.0)), multipv=3)
        print("Top engine choices:")
        for i, info in enumerate(infos, 1):
            temp, san_line = board.copy(), []
            for mv in info.get("pv", [])[:5]:
                san_line.append(temp.san(mv)); temp.push(mv)
            print(f"{i}. {info['score'].pov(board.turn)} | {' '.join(san_line)}")

def engine_move():
    if board.is_game_over(claim_draw=True) or board.turn == human_color: return
    result = engine.play(board, chess.engine.Limit(time=float(move_time.value)))
    san = board.san(result.move); board.push(result.move)
    render_board(f"Engine played {san}"); show_analysis()

def start_game(_=None):
    global board, human_color
    board = chess.Board()
    human_color = chess.WHITE if side.value == "white" else chess.BLACK
    render_board("New game started")
    if human_color == chess.BLACK: engine_move()
    else: show_analysis()

def play_human_move(_=None):
    text = move_input.value.strip().lower(); move_input.value = ""
    if board.is_game_over(claim_draw=True): return render_board("Game already over")
    if board.turn != human_color: return render_board("Wait for the engine")
    try: move = chess.Move.from_uci(text)
    except Exception: return render_board("Use UCI notation, for example e2e4")
    if move not in board.legal_moves: return render_board(f"Illegal move: {text}")
    san = board.san(move); board.push(move); render_board(f"You played {san}"); engine_move()

def undo_turn(_=None):
    if len(board.move_stack) >= 2:
        board.pop(); board.pop(); render_board("Last turn undone"); show_analysis()
    else: render_board("Nothing to undo")

def resign(_=None):
    winner = "Black" if human_color == chess.WHITE else "White"
    status.value = f"<b>You resigned. {winner} wins.</b>"

new_button.on_click(start_game); play_button.on_click(play_human_move)
undo_button.on_click(undo_turn); resign_button.on_click(resign)
try: move_input.on_submit(play_human_move)
except Exception: pass

display(widgets.VBox([
    widgets.HBox([side, move_time]),
    widgets.HBox([move_input, play_button]),
    widgets.HBox([new_button, undo_button, resign_button]), status
]), board_output, analysis_output)
start_game()


## 5. Close the engine when finished


In [ ]:
engine.quit()
print("Engine closed.")


## Architecture

- **Strength:** Stockfish NNUE
- **Chess rules and UCI:** python-chess
- **Notebook GUI:** ipywidgets
- **Runtime:** Databricks or Jupyter CPU
- **Future research:** a TensorFlow policy/value network can be trained separately and compared with Stockfish, but a small notebook-trained network should not be presented as super-GM strength.
